# Atelier 2 — CNN — adaptation KARMAS Hammam

Source : [Eachr4f/atelier2-mnist](https://github.com/Eachr4f/atelier2-mnist),
commit `47654402f475fbe0e1d608b8e8417f28bb080d27`. Les originaux exécutés sont conservés dans
[`references/eachr4f/notebooks/`](../../references/eachr4f/notebooks/).

**Cette version est corrigée et n'a pas encore été exécutée sur GPU.** Ses sorties
ont été effacées pour ne pas attribuer les anciens scores au code modifié.
Dans Colab : activer un GPU puis **Tout exécuter**. Un ou deux GPU sont acceptés.

Adaptations : chemins portables et téléchargement vérifié ; partage commun
54 000 train / 6 000 validation / 10 000 test (seed 42) ; export des nouveaux résultats.
Faster R-CNN : BatchNorm figées en validation, absence de détection = -1 et loss test
explicitement diagnostique. ViT : vraie validation, checkpoint sélectionné sur
validation et moyenne de loss test pondérée par les effectifs des lots.

Les architectures et réglages de base proviennent du dépôt attribué, avec une
convolution CNN à trois blocs, des têtes compactes pour VGG16/AlexNet et le ViT
pédagogique à 8 dimensions. Le partage et les corrections changent le protocole :
**ne pas réutiliser les scores d'origine comme résultats de cette adaptation.**

In [ ]:

from pathlib import Path
import os, sys, subprocess, importlib.util
ROOT = Path.cwd()
if not (ROOT / 'src/adaptation.py').exists():
    # Accepter un lancement depuis notebooks/colab/ ou depuis Colab lui-même.
    for candidate in list(ROOT.parents):
        if (candidate / 'src/adaptation.py').exists():
            ROOT = candidate
            break
    else:
        ROOT = Path('/content/Atelier2_DeepLearning') if Path('/content').exists() else Path('/kaggle/working/Atelier2_DeepLearning')
        if not ROOT.exists():
            subprocess.run(['git', 'clone', 'https://github.com/KarmasHammam/Atelier2_DeepLearning.git', str(ROOT)], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['PYTHONPATH'] = str(ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
if importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Active un GPU : Colab > Exécution > Modifier le type d’exécution > GPU.')
from src.adaptation import prepare_work, save_result
WORK_DIR = prepare_work('cnn')
os.environ['ATELIER2_WORK_DIR'] = str(WORK_DIR)
print('Adaptation KARMAS Hammam ; source Eachr4f, commit 47654402f475fbe0e1d608b8e8417f28bb080d27')
print('Matériel de cette session :', torch.cuda.get_device_name(0))
print('Les résultats de référence importés ne sont pas ceux de cette nouvelle exécution.')


# Lab 2 — Partie 1 : CNN classifier pour MNIST

Notebook PyTorch destiné à Kaggle avec GPU (T4). Il entraîne un CNN, suit la perte et l’accuracy sur validation, conserve le meilleur modèle et applique l’early stopping. Le jeu de test n’est utilisé qu’à la fin.

**Avant de lancer :** dans Kaggle, ajoute le dataset `hojjatk/mnist-dataset` via **Add Input**, puis active **Settings → Accelerator → GPU T4**.


In [ ]:
from pathlib import Path
import os, gzip, shutil, random, copy, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("GPU non disponible. Activez GPU T4 dans les paramètres du notebook Kaggle.")
device = torch.device("cuda")
print("GPU :", torch.cuda.get_device_name(0))


## Préparer les fichiers MNIST

Cette cellule recherche automatiquement les quatre fichiers IDX classiques dans `/kaggle/input`, qu’ils soient compressés en `.gz` ou non, puis les place dans le dossier attendu par torchvision.


In [ ]:
# Les IDX MNIST ont été téléchargés et vérifiés par prepare_work.
print('Données locales :', WORK_DIR / 'MNIST/raw')

## Charger et séparer les données

La validation contient 6 000 images du train officiel. Les transformations aléatoires légères ne sont appliquées qu’aux images d’entraînement.


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomAffine(degrees=8, translate=(0.08, 0.08)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Deux instances distinctes permettent d'appliquer des transforms différents
full_train_aug = datasets.MNIST(str(WORK_DIR / ''), train=True, download=False, transform=train_transform)
full_train_eval = datasets.MNIST(str(WORK_DIR / ''), train=True, download=False, transform=eval_transform)
indices = torch.randperm(len(full_train_aug), generator=torch.Generator().manual_seed(SEED)).tolist()
val_size = 6000
val_indices, train_indices = indices[:val_size], indices[val_size:]

train_set = torch.utils.data.Subset(full_train_aug, train_indices)
val_set = torch.utils.data.Subset(full_train_eval, val_indices)
test_set = datasets.MNIST(str(WORK_DIR / ''), train=False, download=False, transform=eval_transform)

BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_set)} | validation: {len(val_set)} | test: {len(test_set)}")


In [ ]:
import matplotlib.pyplot as plt

# Échantillon de 8 images du jeu d'entraînement
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    # Annuler la normalisation pour afficher l'image en niveaux de gris
    image_to_show = image.squeeze(0) * 0.3081 + 0.1307
    ax.imshow(image_to_show.numpy(), cmap='gray')
    ax.set_title(f"Label réel : {label.item()}")
    ax.axis('off')
plt.suptitle('Échantillon d’images MNIST (entraînement)')
plt.tight_layout()
plt.show()


## Définir le CNN

Le réseau utilise trois convolutions, BatchNorm, ReLU, max pooling, dropout et deux couches entièrement connectées. La dernière couche renvoie 10 logits, un par chiffre.


In [ ]:
class MNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                    # 32 x 14 x 14
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                    # 64 x 7 x 7
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Dropout2d(0.15)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.40),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = MNIST_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
print(model)


## Entraînement avec early stopping

Le meilleur état du modèle est choisi sur la perte de validation. L’entraînement s’arrête après plusieurs époques sans amélioration.


In [ ]:
def run_epoch(loader, training=False):
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

EPOCHS = 30
PATIENCE = 6
best_val_loss = float('inf')
best_state = None
no_improve = 0
history = []
start = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)
    scheduler.step(val_loss)
    history.append((epoch, train_loss, train_acc, val_loss, val_acc))
    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f}, acc {train_acc:.4f} | val loss {val_loss:.4f}, acc {val_acc:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
    if no_improve >= PATIENCE:
        print('Early stopping.')
        break

model.load_state_dict(best_state)
training_seconds = time.time() - start
print(f"Temps d'entraînement : {training_seconds:.1f} secondes")


In [ ]:
# Tracer loss et accuracy pour entraînement et validation
epochs = [row[0] for row in history]
train_losses = [row[1] for row in history]
train_accuracies = [row[2] for row in history]
val_losses = [row[3] for row in history]
val_accuracies = [row[4] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs, train_losses, marker='o', label='Train')
axes[0].plot(epochs, val_losses, marker='o', label='Validation')
axes[0].set_title('Loss par époque')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(epochs, train_accuracies, marker='o', label='Train')
axes[1].plot(epochs, val_accuracies, marker='o', label='Validation')
axes[1].set_title('Accuracy par époque')
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.8, 1.0)
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()


## Évaluation finale sur le jeu de test

Accuracy, F1 macro, perte et rapport par classe sont calculés avec le meilleur modèle sauvegardé.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

model.eval()
all_preds, all_labels = [], []
test_loss_total, n_test = 0.0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels_gpu = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels_gpu)
        test_loss_total += loss.item() * labels.size(0)
        n_test += labels.size(0)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())

test_loss = test_loss_total / n_test
test_accuracy = accuracy_score(all_labels, all_preds)
test_f1_macro = f1_score(all_labels, all_preds, average='macro')
print(f"Temps d'entraînement : {training_seconds:.1f} s")
print(f"Test loss            : {test_loss:.4f}")
print(f"Test accuracy        : {test_accuracy:.4f}")
print(f"Test F1 macro        : {test_f1_macro:.4f}")
print('\nRapport par classe :')
print(classification_report(all_labels, all_preds, digits=4))

model_path = str(WORK_DIR / 'mnist_cnn_best.pth')
torch.save(model.state_dict(), model_path)
print('Modèle sauvegardé :', model_path)


In [ ]:
# Afficher quatre images du test et la prédiction du modèle
images, labels = next(iter(test_loader))
images_gpu = images[:4].to(device)
with torch.no_grad():
    predictions = model(images_gpu).argmax(dim=1).cpu()

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    image_to_show = images[i].squeeze(0) * 0.3081 + 0.1307
    pred = predictions[i].item()
    true = labels[i].item()
    color = 'green' if pred == true else 'red'
    ax.imshow(image_to_show.numpy(), cmap='gray')
    ax.set_title(f"Prédit : {pred}\nRéel : {true}", color=color)
    ax.axis('off')
plt.suptitle('Quatre prédictions sur le jeu de test (vert = correct, rouge = incorrect)')
plt.tight_layout()
plt.show()


## Interprétation

Compare les courbes train/validation et les scores finaux. Si l’accuracy train continue d’augmenter alors que la perte validation remonte, cela indique un début de surapprentissage ; l’early stopping restaure alors l’état ayant obtenu la meilleure perte de validation.


## Enregistrer cette nouvelle exécution

Les métriques sont calculées depuis les prédictions de cette session GPU. Le dossier
`results/adapted_gpu/` reste distinct des résultats importés et des anciens profils.
Enregistrer aussi le notebook dans GitHub **avec ses sorties**. L'archive permet de
conserver les résultats, même si les cinq modèles ne sont pas tous terminés.

In [ ]:
result = save_result('cnn', all_labels, all_preds, test_loss, training_seconds, WORK_DIR)
from scripts.export_adapted_results import export
archive = export(allow_partial=True)
print('Archive :', archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Archive disponible localement.')